In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from scipy.ndimage import gaussian_filter
from utils import *

In [2]:
folder = 'temp/'
files = sorted(glob.glob(folder + '*.fits'))
files

['temp/phi-fdt-flat_20260310T040003_V202607141542C_0663100100.fits',
 'temp/phi-fdt-ghost_20260310T040003_V202607141542C_0663100100.fits']

In [3]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
flat_file, ghost_file = files

with fits.open(dark_file) as hdul:
    dark = hdul[0].data

with fits.open(flat_file) as hdul:
    flat = hdul[0].data

with fits.open(ghost_file) as hdul:
    ghost = hdul[0].data

ghost = demodulate(ghost)

In [4]:
def get_wv_shift(data, cpos=5, pol=0, delta_wv=0.069):
    temp = data.copy().reshape((6, -1, data.shape[-2], data.shape[-1]))[:,pol]
    temp = np.delete(temp, cpos, axis=0)

    t = np.argmin(temp, axis=0)
    l, a, r = np.take_along_axis(temp, np.array([(t - 1) % 5, t, (t + 1) % 5]), axis=0)
    b, c = (r - l) / 2, (l + r) - 2 * a

    with np.errstate(invalid='ignore'):
        return (t - 2 - b / c) * delta_wv


def calc_ghost_scaling(data, cpos=5, delta_wv=0.069, sigma=0.043, gamma=0.053, depth=0.66):
    from scipy.signal import convolve

    dx = 0.001
    x = np.arange(-10,10 + dx / 2, dx)
    f = 1 - depth * np.exp(-x ** 2 / 2 / sigma ** 2)
    g = 1 / (1 + (x / gamma) ** 2)
    q = 2 * (1 - convolve(f, g ** 2, mode='same') / convolve(f, g, mode='same'))

    wv = np.arange(-2,3) * delta_wv
    shift = get_wv_shift(data, cpos=cpos, delta_wv=delta_wv)

    Q = interpolate(q.reshape(-1,1,1), x, wv.reshape(-1,1,1) - np.expand_dims(shift, axis=0))
    return np.insert(np.nan_to_num(Q, nan=1), cpos, np.ones_like(shift), axis=0)

In [5]:
#folder = '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/'
folder = '/home/ulyanov/data/solo/phi/test/'
files = sorted(glob.glob(folder + '*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240101T040003_V202401090117C_0441010503.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240106T210007_V202401100517C_0441060508.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240107T000009_V202401110118C_0441070501.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240207T000009_V202402130123C_0442070501.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240207T060009_V202402130123C_0442070502.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240318T190009_V202405151841C_0443180504.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240328T060009_V202405152307C_0443281521.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240330T044009_V202405152319C_0443301531.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240330T233009_V202405152329C_0443300501.fits.gz',
 '/home/ulyanov/data/solo/ph

In [6]:
with fits.open(files[-2]) as hdul:
    header = hdul[0].header
    data = hdul[0].data

xr, yr = reflection_point_predict(header)
cpos = header['CONTPOS'] - 1
print(cpos)

nx, ny = data.shape[-2:]

data = data.reshape(6,4,nx,ny)
data -= crop(dark, header) * 0.4
data /= crop(flat, header)

q = calc_ghost_scaling(data, cpos=cpos)

0


In [7]:
i = cpos

temp = data[i].copy()
temp = realign(temp)
temp = demodulate(temp)

reflection = reflect(gaussian_filter(temp[0], 8), xr, yr)

In [8]:
mask = (temp[0] > 1000) * (reflection < 1000)
np.median(temp[1][mask]), np.median(temp[2][mask]), np.median(temp[3][mask])

(np.float64(4.16435620744187),
 np.float64(2.763307510997265),
 np.float64(1.5697323474596487))

In [11]:
j = 3

plt.figure(figsize=(10,10))
plt.imshow(temp[j] - crop(ghost, header)[j] * reflection * q[i], vmin=-30, vmax=30)
plt.tight_layout()

In [13]:
plt.figure(figsize=(10,10))
plt.imshow(temp[1] + temp[2], vmin=-30, vmax=30)
plt.tight_layout()